
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.5_flash_attention_problem/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.5_flash_attention_problem/lab.ipynb)

# Lab 3.5: The Standard Attention Problem

We measure three concrete problems with standard attention:
1. Memory grows O(N²) with sequence length
2. Wall-clock time is slow
3. The operation is memory-bound (low arithmetic intensity)

In [ ]:
import torch
import time
import matplotlib.pyplot as plt
import numpy as np

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Memory Growth is O(N²)

Standard attention computes `S = Q @ K^T`, producing an N×N score matrix.
We allocate this matrix at increasing sequence lengths and measure GPU memory.

In [ ]:
# Measure memory consumed by the NxN attention score matrix
seq_lengths = [512, 1024, 2048, 4096, 8192]
d_head = 128  # typical head dimension
memory_mb = []

for N in seq_lengths:
    # Each element is float16 = 2 bytes; matrix is NxN
    score_matrix_bytes = N * N * 2
    memory_mb.append(score_matrix_bytes / (1024**2))
    # Verify by actually allocating on device
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        S = torch.empty(N, N, dtype=torch.float16, device=device)
        actual_mb = torch.cuda.max_memory_allocated() / (1024**2)
        del S
        print(f'N={N:>5}: theoretical={memory_mb[-1]:.1f} MB, actual={actual_mb:.1f} MB')
    else:
        print(f'N={N:>5}: {memory_mb[-1]:.1f} MB (computed, no GPU)')

# Plot: memory vs sequence length
fig_1, ax_1 = plt.subplots(figsize=(8, 4))
ax_1.plot(seq_lengths, memory_mb, 'o-', color='#e63946', linewidth=2)
ax_1.set_xlabel('Sequence Length (N)')
ax_1.set_ylabel('Score Matrix Memory (MB)')
ax_1.set_title('Attention Score Matrix: O(N²) Memory Growth')
ax_1.set_xticks(seq_lengths)
ax_1.grid(True, alpha=0.3)
# Annotate the quadratic relationship
ax_1.annotate(f'{memory_mb[-1]:.0f} MB for one head!',
            xy=(seq_lengths[-1], memory_mb[-1]),
            xytext=(seq_lengths[-2], memory_mb[-1]*0.8),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10)
plt.tight_layout()
plt.show()
print(f'\nAt N=8192 with 32 heads: {memory_mb[-1]*32:.0f} MB just for attention scores')

## 2. Standard Attention is Slow

We time the full standard attention: `softmax(Q @ K^T / sqrt(d)) @ V`.

In [ ]:
# Time standard attention at different sequence lengths
d_head_timing = 128
n_heads = 32
test_lengths = [512, 1024, 2048, 4096]
times_ms = []

for N in test_lengths:
    # Single head attention for clarity
    Q = torch.randn(N, d_head_timing, device=device, dtype=torch.float16)
    K = torch.randn(N, d_head_timing, device=device, dtype=torch.float16)
    V = torch.randn(N, d_head_timing, device=device, dtype=torch.float16)

    # Warmup
    for _ in range(3):
        S = Q @ K.T / (d_head_timing ** 0.5)
        P = torch.softmax(S, dim=-1)
        O = P @ V
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Timed runs
    start = time.perf_counter()
    n_runs = 10
    for _ in range(n_runs):
        S = Q @ K.T / (d_head_timing ** 0.5)
        P = torch.softmax(S, dim=-1)
        O = P @ V
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_runs * 1000  # ms
    times_ms.append(elapsed)
    print(f'N={N:>5}: {elapsed:.2f} ms per attention (single head)')
    del Q, K, V, S, P, O

# Plot timing
fig_2, ax_2 = plt.subplots(figsize=(8, 4))
ax_2.bar(range(len(test_lengths)), times_ms, color='#457b9d', width=0.6)
ax_2.set_xticks(range(len(test_lengths)))
ax_2.set_xticklabels([str(n) for n in test_lengths])
ax_2.set_xlabel('Sequence Length (N)')
ax_2.set_ylabel('Time (ms)')
ax_2.set_title('Standard Attention Wall-Clock Time (Single Head)')
ax_2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3. Standard Attention is Memory-Bound

Arithmetic intensity = FLOPs / Bytes moved.  
If this ratio is low, the GPU compute units sit idle waiting for data from HBM.

In [ ]:
# Compute arithmetic intensity for standard attention
d = 128  # head dimension

seq_range = np.array([512, 1024, 2048, 4096, 8192, 16384])
intensities = []

for N in seq_range:
    # FLOPs: Q@K^T = 2*N*N*d, P@V = 2*N*N*d, softmax ~ 5*N*N
    flops = 2*N*N*d + 2*N*N*d + 5*N*N

    # Bytes: read Q,K,V (3*N*d*2), write S (N*N*2), read S for softmax (N*N*2),
    #        write P (N*N*2), read P+V for P@V (N*N*2 + N*d*2), write O (N*d*2)
    bytes_moved = (3*N*d*2) + (N*N*2)*3 + (N*d*2) + (N*d*2)

    intensity = flops / bytes_moved
    intensities.append(intensity)

# A100 has ~2000 TFLOPS FP16 and ~2000 GB/s HBM bandwidth
# Ridge point = 2000e12 / 2000e9 = 1000 FLOPs/byte
ridge_point = 1000

fig_3, ax_3 = plt.subplots(figsize=(8, 4))
ax_3.plot(seq_range, intensities, 'o-', color='#2a9d8f', linewidth=2, label='Standard Attention')
ax_3.axhline(y=ridge_point, color='#e63946', linestyle='--', linewidth=1.5, label=f'A100 Ridge Point ({ridge_point} FLOP/byte)')
ax_3.set_xlabel('Sequence Length (N)')
ax_3.set_ylabel('Arithmetic Intensity (FLOPs/byte)')
ax_3.set_title('Standard Attention: Memory-Bound at All Sequence Lengths')
ax_3.legend()
ax_3.grid(True, alpha=0.3)
ax_3.set_yscale('log')
ax_3.set_xscale('log')
plt.tight_layout()
plt.show()

print(f'\nArithmetic intensity at N=4096: {intensities[3]:.1f} FLOP/byte')
print(f'A100 ridge point: {ridge_point} FLOP/byte')
print(f'Ratio: {intensities[3]/ridge_point:.3f}x (well below ridge -> memory-bound)')

## Summary

| Problem | Evidence |
|---------|----------|
| O(N²) memory | 8192 seq len × 32 heads = ~4 GB just for scores |
| Slow | Quadratic time growth with sequence length |
| Memory-bound | Arithmetic intensity far below GPU ridge point |

The bottleneck is clear: standard attention reads and writes the full N×N matrix to HBM repeatedly, and the GPU's compute capability is wasted waiting for memory.

**The next module shows how FlashAttention solves this.**